In [ ]:
import numpy as np
import pandas as pd
import urllib
import json
from pathlib import Path
%matplotlib inline

## Import metadata from Google Sheets

* [Access Google Sheet](https://docs.google.com/spreadsheets/d/1YklUMHrcqeEHhJtmiXgvCnb2B_SaIt99sZ-T6u1Kszc/edit#gid=0)

In [ ]:
sheet_id = "1YklUMHrcqeEHhJtmiXgvCnb2B_SaIt99sZ-T6u1Kszc"
sheet_name = urllib.parse.quote("Raw data v2")
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(url)

In [ ]:
# remove empty columns

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [ ]:
df["nDataType"]

In [ ]:
df.columns

In [ ]:
!head NEXUS_R13_unblinded_raw_files.txt

## Data and functions used for conversion

In [ ]:
facilities = {25: "NEXUS"}

In [ ]:
## From previous analysis
n_data_types = {
    "Test": -1,
    "Background": 0,
    "Co": 1,
    "Co LowR": 2,
    "Cf": 3,
    "Rand": 4,
    "Mon": 7,
    "Cs source": 8,
    "Ba": 9,
    "YBe": 12,
    "SbBe": 13,
    "Y Blank": 14,
    "Sb Blank": 15,
    "Laser": 16,
    "Beam": 17,
    "Beam + Laser": 18,
    "Fe55 source": 19,
    "Fe55": 19,
    "Co57": 20,
    "IV curve": 100,
    "dIdV": 101,
    "NM noise": 102,
    "SC noise": 103,
    "TS noise": 104,
}

## New for Nexus 13

n_data_types["Cs-137"] = 8 # same as "Cs source"
n_data_types["noSource"] = 0 # like "Low Background"

In [ ]:
def extract_nDump(filename):
    return int(filename.split("/")[-1].split("F")[1].split(".")[0])

In [ ]:
assert extract_nDump('/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw/25220319_095807/25220319_095807_F1200.mid.gz') == 1200

## Only register unblinded data

In [ ]:
len(df)

In [ ]:
df = df[df["Path"].str.contains("unblinded")]

In [ ]:
len(df)

## List file in series

In [ ]:
file_list = pd.read_table("NEXUS_R13_unblinded_raw_files.txt", squeeze=False, header=None, names=["file_path"])

In [ ]:
file_list[["path", "series", "file_name"]] = file_list["file_path"].str.rsplit("/", n=2, expand=True)

In [ ]:
file_list.head()

In [ ]:
len(file_list)

In [ ]:
file_list_grouped = file_list.groupby(["path", "series"])

### Statistics about the series

Number of files per series

In [ ]:
file_list_grouped["file_name"].count().hist();

In [ ]:
file_list_grouped["file_name"].count().describe()

In [ ]:
def list_file(path, series):
    """List files at path
    
    This works without actual access to the filesystem, reads from pregenerated text file"""
    return file_list_grouped.get_group((path, series))["file_path"].to_list()

In [ ]:
assert len(list_file("/sdf/group/supercdms/data/CDMS/NEXUS/R13_unblinded/Raw", "25220329_143754")) == 394

## Iterate by row and register the data

* Iterate through all rows in the Pandas Dataframe
* Build a dictionary of processed metadata
* Call `ContinuousRawData` with the `record` dictionary as arguments

In [ ]:
records = []
for row in df.to_dict(orient="records"):
    record = dict(
        filePath=row.pop("Path"),
        facility=facilities[row.pop("Facility")],
        nFridgeRun=row.pop("nFridgeRun"),
        nDataType=n_data_types.get(row.pop("nDataType"), np.nan),
        series=row.pop("Series"),
        nIsJunk=0,
        fileFormat="root"
    )
    record["commentStart"] = [json.dumps(row)]
    for file in list_file(record["filePath"], record["series"]):
        record["fileName"] = file
        record["nDumpNum"] = extract_nDump(file)
        records.append(record.copy())

In [ ]:
len(records)

In [ ]:
records[0]

In [ ]:
records[1]

In [ ]:
records_df =pd.DataFrame(records)

records_df.to_csv("Nexus_13_registration_unblinded.csv", index=False)

In [ ]:
records_df.groupby("series")["filePath"].count()

The CSV is saved into https://docs.google.com/spreadsheets/d/16s31Vvu72A12LDkZBd7IkDK1B3GbMGK0d8ymli7RSY4/edit#gid=962568114